# Who&When: data analysis

This notebook explains the dataset the project uses: **58 failed agent runs from Who&When (Hand-Crafted)**, each labelled with the **Decisive step**, the step where the run went wrong. The project tests whether a prover-estimator debate can find that step, so a person checks one step instead of the whole run.

**How to run it**
- **On Kaggle:** turn **Internet on** in the notebook settings (no GPU needed), then *Run All*. The first cell clones the project from GitHub (branch `SC.V1`). If the repository is private, add a Kaggle Secret named `GITHUB_TOKEN` first.
- **On a Mac:** open it in VS Code from the project folder and *Run All*. It uses the project folder it sits in.

Every figure is also saved as a PNG in `figures/`, for the report.

## 0. Setup: get the project and its code

In [ ]:
import math
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_ADDRESS = "github.com/shoimya/can-debate-be-used-to-reduce-human-review-time.git"
BRANCH = "SC.V1"
KAGGLE_PROJECT_FOLDER = Path("/kaggle/working/can-debate-be-used-to-reduce-human-review-time")


def github_address_with_token_if_any() -> str:
    """Return the repository address, with the Kaggle Secret GITHUB_TOKEN in it if one was added (for a private repo)."""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        return f"https://{token}@{REPOSITORY_ADDRESS}"
    except Exception:
        return f"https://{REPOSITORY_ADDRESS}"


def find_or_clone_project_folder() -> Path:
    """On Kaggle, clone (or update) the project from GitHub; on a Mac, use the folder this notebook sits in."""
    running_on_kaggle = Path("/kaggle/working").exists()
    if not running_on_kaggle:
        return Path.cwd()

    if KAGGLE_PROJECT_FOLDER.exists():
        subprocess.run(["git", "-C", str(KAGGLE_PROJECT_FOLDER), "pull"], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, github_address_with_token_if_any(),
                        str(KAGGLE_PROJECT_FOLDER)], check=True)
    return KAGGLE_PROJECT_FOLDER


# Here we move into the project folder and let Python import its files, so this notebook uses the same data.py.
PROJECT_FOLDER = find_or_clone_project_folder()
os.chdir(PROJECT_FOLDER)
sys.path.insert(0, str(PROJECT_FOLDER))

import data

print("Project folder:", PROJECT_FOLDER)

: 

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Here we fix one colour per agent, so every figure uses the same colours.
AGENT_COLOURS = {
    "human": "#7f7f7f",
    "Orchestrator": "#1f77b4",
    "WebSurfer": "#ff7f0e",
    "FileSurfer": "#2ca02c",
    "Assistant": "#9467bd",
    "ComputerTerminal": "#8c564b",
}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

# Here we create the folder the figures are saved to.
FIGURES_FOLDER = PROJECT_FOLDER / "figures"
FIGURES_FOLDER.mkdir(exist_ok=True)


def save_figure(figure, file_name: str) -> None:
    """Save a figure as a PNG in figures/, sized for a report."""
    figure.savefig(FIGURES_FOLDER / file_name, dpi=200, bbox_inches="tight")

## 1. How many runs are there, and how many are usable?

A run is **usable** if its labelled Decisive step is one of its steps. Without that, a method's answer cannot be scored.

In [ ]:
all_runs = data.download_who_and_when()
usable_runs = data.remove_bad_runs(all_runs)

pd.DataFrame(
    {"Runs": [len(all_runs), len(usable_runs), len(all_runs) - len(usable_runs)]},
    index=["Downloaded", "Usable (Decisive step exists)", "Removed"],
)

> **Note for the report.** The same dataset is also published as a Parquet file. There, 3 runs (rows 25, 39, 50) have a Decisive step that points past the last step. The JSON files used here have the same steps but corrected labels for those 3 runs (`53.json`, `34.json`, `54.json`), so all 58 runs are usable.

### One table with one row per run

Every figure below is drawn from this table.

- **Reading minutes** is an estimate: characters ÷ 6 ≈ words, read at 200 words per minute. The team's own timings (human review set) will replace it.
- **Relative position** of the Decisive step: 0 = first step, 1 = last step.
- **Minutes to reach the mistake**: estimated reading time from the first step up to and including the Decisive step, for a person reading from the start.
- **Agent at label matches** compares the agent who wrote the labelled step with the labelled agent (ignoring capital letters).

In [ ]:
CHARACTERS_PER_WORD = 6
WORDS_PER_MINUTE = 200


def agent_name(role: str) -> str:
    """Return the agent's name without the extra part, for example "Orchestrator (thought)" -> "Orchestrator"."""
    return role.split(" (")[0]


def run_summary(run_number: int, run: data.Trajectory) -> dict:
    """Return the numbers this notebook uses for one run."""
    number_of_steps = len(run.steps)
    characters = sum(len(step.text) for step in run.steps)
    agent_at_label = agent_name(run.steps[run.decisive_step].agent)
    return {
        "run": run_number,
        "file": run.file_name,
        "steps": number_of_steps,
        "characters": characters,
        "reading_minutes": characters / CHARACTERS_PER_WORD / WORDS_PER_MINUTE,
        "decisive_step": run.decisive_step,
        # Here we turn the step number into a position from 0 (first step) to 1 (last step).
        "relative_position": run.decisive_step / (number_of_steps - 1),
        "steps_after_mistake": number_of_steps - 1 - run.decisive_step,
        # Here we add up the text from the first step to the Decisive step: what a person reads before reaching it.
        "minutes_to_reach_mistake": sum(len(step.text) for step in run.steps[:run.decisive_step + 1])
        / CHARACTERS_PER_WORD / WORDS_PER_MINUTE,
        "labelled_agent": run.decisive_agent,
        "agent_at_label": agent_at_label,
        "agent_at_label_matches": agent_at_label.lower() == run.decisive_agent.lower(),
    }


runs_table = pd.DataFrame([run_summary(run_number, run) for run_number, run in enumerate(usable_runs, start=1)])
runs_table.head()

## 2. What does one run look like?

Each bar is one step. Its height is how much text the step contains (log scale, because a step can be one line or a whole web page). The colour is the agent. The red arrow marks the labelled Decisive step. Different Orchestrator steps, such as `Orchestrator (thought)` and `Orchestrator (-> WebSurfer)`, share one colour.

The pilot run is shown if `data/splits/pilot.json` exists; otherwise run 1.

In [ ]:
import json


def run_number_to_show() -> int:
    """Return the pilot run's number if a pilot split exists, otherwise 1."""
    pilot_file = data.SPLITS_FOLDER / "pilot.json"
    if pilot_file.exists():
        return json.loads(pilot_file.read_text())[0]["number"]
    return 1


run_to_show = run_number_to_show()
shown_run = usable_runs[run_to_show - 1]

step_numbers = [step.index for step in shown_run.steps]
step_lengths = [len(step.text) for step in shown_run.steps]
step_colours = [AGENT_COLOURS.get(agent_name(step.agent), "black") for step in shown_run.steps]

figure, axes = plt.subplots(figsize=(12, 3.8))
axes.bar(step_numbers, step_lengths, color=step_colours, width=0.85)
axes.set_yscale("log")
axes.set_xlabel("Step number")
axes.set_ylabel("Characters in the step (log scale)")
axes.set_title(f"Run {run_to_show} ({shown_run.file_name}): {len(shown_run.steps)} steps, "
               f"Decisive step {shown_run.decisive_step} by {shown_run.decisive_agent}")

# Here we mark the Decisive step with an arrow. The label sits above the tallest bar, so it never covers a bar.
tallest_bar = max(step_lengths)
axes.set_ylim(top=tallest_bar * 40)
axes.annotate("Decisive step", xy=(shown_run.decisive_step, step_lengths[shown_run.decisive_step]),
              xytext=(shown_run.decisive_step, tallest_bar * 8), ha="center", color="red", fontweight="bold",
              arrowprops={"arrowstyle": "->", "color": "red", "lw": 1.5})

# Here we add a legend with only the agents that appear in this run.
agents_in_run = sorted({agent_name(step.agent) for step in shown_run.steps})
legend_patches = [plt.Rectangle((0, 0), 1, 1, color=AGENT_COLOURS.get(agent, "black")) for agent in agents_in_run]
axes.legend(legend_patches, agents_in_run, ncol=len(agents_in_run), loc="upper center",
            bbox_to_anchor=(0.5, -0.2), frameon=False)
save_figure(figure, "1_one_trajectory.png")
plt.show()

In [ ]:
def show_steps_around_the_mistake(run: data.Trajectory, steps_before_and_after: int = 3) -> pd.DataFrame:
    """Return a short table of the steps just before and after the Decisive step, with each step's first words."""
    first_step = max(0, run.decisive_step - steps_before_and_after)
    last_step = min(len(run.steps) - 1, run.decisive_step + steps_before_and_after)
    rows = []
    for step in run.steps[first_step:last_step + 1]:
        rows.append({
            "step": step.index,
            "agent": step.agent,
            "first words": " ".join(step.text.split())[:140] + " ...",
            "": "<- Decisive step" if step.index == run.decisive_step else "",
        })
    return pd.DataFrame(rows).set_index("step")


print("Question:", shown_run.question)
print("Why this step (annotators):", shown_run.decisive_reason)
pd.set_option("display.max_colwidth", 160)
show_steps_around_the_mistake(shown_run)

## 3. How long are the runs?

Length decides how much work a person has without help, and how many rounds the debate needs.

In [ ]:
figure, (steps_axes, minutes_axes) = plt.subplots(1, 2, figsize=(12, 3.8))

steps_axes.hist(runs_table["steps"], bins=range(0, 140, 10), color="#4c72b0", edgecolor="white")
steps_axes.axvline(runs_table["steps"].median(), color="black", linestyle="--",
                   label=f"median = {runs_table['steps'].median():.1f} steps")
steps_axes.set_xlabel("Steps in the run")
steps_axes.set_ylabel("Number of runs")
steps_axes.set_title("Run length in steps")
steps_axes.legend(frameon=False)

minutes_axes.hist(runs_table["reading_minutes"], bins=12, color="#55a868", edgecolor="white")
minutes_axes.axvline(runs_table["reading_minutes"].median(), color="black", linestyle="--",
                     label=f"median = {runs_table['reading_minutes'].median():.0f} min")
minutes_axes.set_xlabel("Estimated minutes to read the whole run")
minutes_axes.set_ylabel("Number of runs")
minutes_axes.set_title("Reading time without help (estimate)")
minutes_axes.legend(frameon=False)

save_figure(figure, "2_run_length.png")
plt.show()

# Here we show the summary numbers. Model tokens are estimated as characters / 4.
length_summary = pd.DataFrame({
    "steps": runs_table["steps"],
    "characters": runs_table["characters"],
    "model tokens (estimate)": runs_table["characters"] / 4,
    "reading minutes (estimate)": runs_table["reading_minutes"],
}).describe().loc[["min", "50%", "mean", "max"]].rename(index={"50%": "median"})
length_summary.round(1)

## 4. How do the runs end?

Every run in this dataset failed. This shows **how**: either the system stopped the team (no agent was chosen to speak, the 30-round limit, or the 1500-second time limit), or the last step was an agent's, usually a wrong final answer. Runs stopped by the round limit are why some runs are exactly 130 steps long.

In [ ]:
def how_the_run_ends(run: data.Trajectory) -> str:
    """Return a short description of how a run ended, read from its last step."""
    last_step = run.steps[-1]
    last_text = " ".join(last_step.text.split())
    if "termination condition" not in last_step.agent:
        return ENDED_BY_AN_AGENT
    if "Max rounds" in last_text:
        return "Stopped: round limit (30 rounds)"
    if "Max time" in last_text:
        return "Stopped: time limit (1500 s)"
    if "No agent selected" in last_text:
        return "Stopped: no agent chosen to speak"
    return "Stopped: other"


ENDED_BY_AN_AGENT = "Ended by an agent's step"
runs_table["ending"] = [how_the_run_ends(run) for run in usable_runs]
ending_counts = runs_table["ending"].value_counts().sort_values()

figure, axes = plt.subplots(figsize=(9, 3))
axes.barh(ending_counts.index, ending_counts.values, color="#4c72b0")
# Here we write each count at the end of its bar.
for bar_position, count in enumerate(ending_counts.values):
    axes.text(count + 0.3, bar_position, str(count), va="center")
axes.set_xlabel("Number of runs")
axes.set_title("How the 58 failed runs end")
save_figure(figure, "3_how_runs_end.png")
plt.show()

## 5. Where in the run is the mistake?

The prover-estimator debate halves the run every round. If mistakes sat mostly in one place, a simple guess would do well and the debate's halving would matter less.

The second figure shows the cost of reading without help: a person reading from the start must read everything up to the Decisive step before they can find it. The dashed line is the median for reading the whole run, for comparison.

In [ ]:
figure, (position_axes, after_axes) = plt.subplots(1, 2, figsize=(12, 3.8))

position_axes.hist(runs_table["relative_position"], bins=[i / 10 for i in range(11)],
                   color="#c44e52", edgecolor="white")
position_axes.set_xlabel("Position of the Decisive step (0 = first step, 1 = last step)")
position_axes.set_ylabel("Number of runs")
position_axes.set_title("Where the mistake is")

minutes_to_mistake = runs_table["minutes_to_reach_mistake"]
# Here we put the rare runs above 60 minutes into one last bar, so one very long run does not squeeze the rest.
LONGEST_MINUTES_SHOWN = 60
after_axes.hist(minutes_to_mistake.clip(upper=LONGEST_MINUTES_SHOWN), bins=range(0, LONGEST_MINUTES_SHOWN + 5, 5),
                color="#8172b2", edgecolor="white")
after_axes.set_xticks(range(0, LONGEST_MINUTES_SHOWN + 1, 10))
after_axes.set_xticklabels([str(tick) for tick in range(0, LONGEST_MINUTES_SHOWN, 10)] + [f"{LONGEST_MINUTES_SHOWN}+"])
after_axes.axvline(minutes_to_mistake.median(), color="black", linewidth=2,
                   label=f"median to reach the mistake = {minutes_to_mistake.median():.0f} min")
after_axes.axvline(runs_table["reading_minutes"].median(), color="black", linestyle="--",
                   label=f"median for the whole run = {runs_table['reading_minutes'].median():.0f} min")
after_axes.set_xlabel("Estimated minutes read from the start until the mistake")
after_axes.set_ylabel("Number of runs")
after_axes.set_title("Reading needed to reach the mistake (estimate)")
# Here we put the legend below the plot, so it never covers a bar or a line.
after_axes.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), frameon=False)

save_figure(figure, "4_where_the_mistake_is.png")
plt.show()

# Here we count runs whose mistake is in the first, middle or last third.
thirds = pd.cut(runs_table["relative_position"], bins=[-0.01, 1 / 3, 2 / 3, 1.0],
                labels=["first third", "middle third", "last third"])
thirds.value_counts().reindex(["first third", "middle third", "last third"]).to_frame("runs")

## 6. Which agents get blamed?

Left bar of each pair: the agent's share of **all steps**. Right bar: its share of **Decisive steps**. An agent whose right bar is taller fails more often than how much it acts would suggest.

In [ ]:
all_step_agents = pd.Series([agent_name(step.agent) for run in usable_runs for step in run.steps])
# Here we write each labelled agent in one spelling (for example "Websurfer" -> "WebSurfer").
known_spellings = {name.lower(): name for name in AGENT_COLOURS}
blamed_agents = runs_table["labelled_agent"].map(lambda name: known_spellings.get(name.lower(), name))

agent_shares = pd.DataFrame({
    "share of all steps": all_step_agents.value_counts(normalize=True),
    "share of Decisive steps": blamed_agents.value_counts(normalize=True),
}).fillna(0).sort_values("share of Decisive steps", ascending=False)

figure, axes = plt.subplots(figsize=(9, 3.8))
agent_shares.plot.bar(ax=axes, color=["#bbbbbb", "#c44e52"], rot=0, width=0.75)
axes.set_ylabel("Share")
axes.set_title("How much each agent acts vs how often it is blamed")
axes.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
axes.legend(frameon=False)
save_figure(figure, "5_who_is_blamed.png")
plt.show()

(agent_shares * 100).round(1).astype(str) + " %"

## 7. How far can we trust the labels?

Three checks:
1. The labelled Decisive step exists in the run (Section 1).
2. The step at the label was written by the labelled agent.
3. Agent names are spelled one way.

Mismatches in check 2 are not always errors: for example, a mistake can be blamed on WebSurfer while the labelled step is the Orchestrator's instruction *to* WebSurfer. They show why results should also be scored with a tolerance of a few steps.

In [ ]:
spelling_variants = (
    runs_table["labelled_agent"].groupby(runs_table["labelled_agent"].str.lower()).unique()
)
spelling_variants = spelling_variants[spelling_variants.map(len) > 1]

label_checks = pd.DataFrame({"runs": [
    len(usable_runs),
    int(runs_table["agent_at_label_matches"].sum()),
    int((~runs_table["agent_at_label_matches"]).sum()),
]}, index=["Decisive step exists", "Agent at the labelled step matches", "Agent does not match"])
print("Agent names spelled more than one way:", [list(variants) for variants in spelling_variants])
label_checks

In [ ]:
# Here we list the runs where the step at the label was written by a different agent.
runs_table.loc[~runs_table["agent_at_label_matches"], ["run", "file", "decisive_step", "labelled_agent", "agent_at_label"]]

## 8. How hard is the task, and what will the debate cost?

**Simple baselines** set the floor any method must beat:
- **Random guess:** pick any step; expected chance of the exact step is 1 ÷ steps.
- **Middle step:** always pick the step in the middle of the run.

Results are shown for the exact step and for **within ±2 steps**, the tolerance the project plans to use.

**Debate size:** the prover-estimator debate halves the run each round, so it needs about log₂(steps) rounds, and about 2 × rounds + 4 model calls (research ticket 09). Every baseline gets the same number of calls (the TA's rule).

In [ ]:
TOLERANCE_IN_STEPS = 2

middle_step = (runs_table["steps"] - 1) // 2
distance_from_middle = (middle_step - runs_table["decisive_step"]).abs()
# Here we work out the chance a random guess lands within ±2 steps: the number of steps in that window ÷ all steps.
steps_in_window = runs_table.apply(
    lambda row: min(row["steps"] - 1, row["decisive_step"] + TOLERANCE_IN_STEPS)
    - max(0, row["decisive_step"] - TOLERANCE_IN_STEPS) + 1, axis=1)

baselines = pd.DataFrame({
    "exact step": [(1 / runs_table["steps"]).mean(), (distance_from_middle == 0).mean()],
    f"within ±{TOLERANCE_IN_STEPS} steps": [(steps_in_window / runs_table["steps"]).mean(),
                                            (distance_from_middle <= TOLERANCE_IN_STEPS).mean()],
}, index=["Random guess (expected)", "Always the middle step"])

debate_rounds = runs_table["steps"].map(lambda steps: math.ceil(math.log2(steps)))
debate_calls = 2 * debate_rounds + 4

print(f"Debate rounds per run: median {debate_rounds.median():.0f}, max {debate_rounds.max()}")
print(f"Model calls per debate (about 2 x rounds + 4): median {debate_calls.median():.0f}, max {debate_calls.max()}")
(baselines * 100).round(1).astype(str) + " %"

## 9. Summary for the report

In [ ]:
report_numbers = pd.Series({
    "Runs (all usable)": f"{len(usable_runs)} of {len(all_runs)}",
    "Steps per run: median (min-max)": f"{runs_table['steps'].median():.1f} ({runs_table['steps'].min()}-{runs_table['steps'].max()})",
    "Reading minutes per run: median (estimate)": f"{runs_table['reading_minutes'].median():.0f}",
    "Reading minutes to reach the mistake: median (estimate)": f"{runs_table['minutes_to_reach_mistake'].median():.0f}",
    "Runs stopped by the system (no agent, round or time limit)":
        f"{(runs_table['ending'] != ENDED_BY_AN_AGENT).sum()} of {len(runs_table)}",
    "Most blamed agent": f"{agent_shares.index[0]} ({agent_shares['share of Decisive steps'].iloc[0]:.0%} of Decisive steps)",
    "Agent at labelled step matches": f"{runs_table['agent_at_label_matches'].sum()} of {len(runs_table)}",
    "Random guess, exact / within ±2": f"{baselines.iloc[0, 0]:.1%} / {baselines.iloc[0, 1]:.1%}",
    "Middle step, exact / within ±2": f"{baselines.iloc[1, 0]:.1%} / {baselines.iloc[1, 1]:.1%}",
    "Debate rounds / model calls per run: median": f"{debate_rounds.median():.0f} / {debate_calls.median():.0f}",
}).to_frame("value")
print("Figures saved in:", FIGURES_FOLDER)
report_numbers